<a href="https://colab.research.google.com/github/AmiraFaisal/ETEC2T/blob/main/ChartJudge_2B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers torch pillow pandas tqdm --quiet

# for ChartGemma

In [ ]:
import json
import os
import re
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm

IMAGES_DIR   = "/content/drive/MyDrive/EvaltheEvaluators/images/"
JSON_PATH    = "/content/drive/MyDrive/EvaltheEvaluators/ChartGemma_Model/ChartGemma_Descriptions.json"
OUTPUT_CSV   = "/content/drive/MyDrive/EvaltheEvaluators/evaluations/ChartJudge_Suite/ChartGemma_pointwise_scores.csv"
METRICS      = ["Factual Correctness", "Relevance", "Fluency"]
MODEL_NAME   = "Qwen/Qwen2-VL-2B-Instruct"

## ---------------------------------- prompt ----------------------------------
PROMPT_TEMPLATE = """<image>

You are an expert chart-caption evaluator. You will be shown a chart image and a caption generated for it.
Your task is to evaluate the caption on a single criterion using a 1–10 Likert scale.

Criterion: {metric}

Scoring guide:
   1–2  = Very poor   (major errors or completely irrelevant)
   3–4  = Poor        (significant issues, limited accuracy or clarity)
   5–6  = Acceptable  (partially correct or relevant, noticeable gaps)
   7–8  = Good        (mostly accurate, relevant, and well-expressed)
   9–10 = Excellent   (fully accurate, highly relevant, fluent and precise)

Caption: {caption}

Evaluate the caption strictly against the chart image shown.
Respond ONLY with valid JSON, no markdown, no extra text:
{{"score": <integer 1-10>, "explanation": "<one sentence reason>"}}"""

## ---------------------------------- set up ----------------------------------
def mount_drive():
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=True)
        print("Google Drive mounted")
    except ImportError:
        print("Not in Colab")


def load_model():
    print(f"\nLoading {MODEL_NAME} ...")
    from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

    processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Model loaded on {device}")
    if device == "cuda":
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  GPU memory: {mem:.1f} GB total")
    return model, processor


def load_data():
    with open(JSON_PATH, "r") as f:
        data = json.load(f)
    print(f"Loaded {len(data)} samples from JSON")
    missing_keys = [i for i, d in enumerate(data) if "answer" not in d or "img_id" not in d]
    if missing_keys:
        print(f"  WARNING: {len(missing_keys)} records missing '{"answer"}' or '{"img_id"}' — will be skipped")
    return data

## ------------------------------- inference ----------------------------------
def parse_response(text: str) -> dict:
    """Extract JSON from model output."""
    text = text.strip()
    # strip markdown fences
    text = re.sub(r"```json\s*", "", text)
    text = re.sub(r"```\s*", "", text)
    # find first {...}
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    # last resort: parse entire string
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"score": None, "explanation": f"PARSE_ERROR: {text[:120]}"}


def score_caption(model, processor, image: Image.Image, caption: str, metric: str) -> dict:
    prompt_text = f"""You are an expert chart evaluator. Rate the following chart caption on a single criterion using a 1-10 Likert scale.

    Criterion: {metric}

    Scoring guide:
      1–2  = Very poor   (major errors or completely irrelevant)
      3–4  = Poor        (significant issues, limited accuracy or clarity)
      5–6  = Acceptable  (partially correct or relevant, noticeable gaps)
      7–8  = Good        (mostly accurate, relevant, and well-expressed)
      9–10 = Excellent   (fully accurate, highly relevant, fluent and precise)

    Caption: {caption}

    Evaluate the caption strictly against the chart image shown.
    Respond ONLY with valid JSON, no markdown, no extra text:
    {{"score": <integer 1-10>, "explanation": "<one sentence reason>"}}"""

    # Qwen2-VL requires messages format with image content block
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]

    # apply chat template
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
        )

    # decode only new tokens
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    response = processor.decode(new_tokens, skip_special_tokens=True)

    return parse_response(response)

## --------------------------- main function ----------------------------------
def run_evaluation():
    mount_drive()
    model, processor = load_model()
    data = load_data()
    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
    rows = []
    skipped = 0

    for sample in tqdm(data, desc="Scoring captions"):
        img_id  = sample.get("img_id")
        caption = sample.get("answer")

        if not img_id or not caption:
            skipped += 1
            continue

        img_path = os.path.join(IMAGES_DIR, f"{img_id}{".png"}")
        if not os.path.exists(img_path):
            print(f"\n  Image not found, skipping: {img_path}")
            skipped += 1
            continue

        image = Image.open(img_path).convert("RGB")

        row = {"img_id": img_id, "caption": caption}

        for metric in METRICS:
            result = score_caption(model, processor, image, caption, metric)
            col = metric.lower().replace(" ", "_")   # e.g. "factual_correctness"
            row[f"{col}_score"]       = result.get("score")
            row[f"{col}_explanation"] = result.get("explanation")

        rows.append(row)
        pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)

    # Final save
    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_CSV, index=False)

    print(f"\n{'='*60}")
    print(f"Results saved to: {OUTPUT_CSV}")
    print(f"\nScore averages:")
    for metric in METRICS:
        col = metric.lower().replace(" ", "_") + "_score"
        if col in df.columns:
            avg = pd.to_numeric(df[col], errors="coerce").mean()
            print(f"  {metric:25s}: {avg:.2f} / 10.00")
    print('='*60)

    return df



if __name__ == "__main__":
    df = run_evaluation()


Mounted at /content/drive
Google Drive mounted

Loading Qwen/Qwen2-VL-2B-Instruct ...


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

Model loaded on cuda
  GPU memory: 15.6 GB total
Loaded 21 samples from JSON


Scoring captions: 100%|██████████| 21/21 [04:19<00:00, 12.36s/it]


Results saved to: /content/drive/MyDrive/EvaltheEvaluators/evaluations/ChartJudge_Suite/ChartGemma_pointwise_scores.csv

Score averages:
  Factual Correctness      : 7.50 / 10.00
  Relevance                : 7.62 / 10.00
  Fluency                  : 7.95 / 10.00


# for Qwen

In [ ]:
import json
import os
import re
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm


IMAGES_DIR   = "/content/drive/MyDrive/EvaltheEvaluators/images/"
JSON_PATH    = "/content/drive/MyDrive/EvaltheEvaluators/Qwen_Model/Qwen_Descriptions.json"
OUTPUT_CSV   = "/content/drive/MyDrive/EvaltheEvaluators/evaluations/ChartJudge_Suite/Qwen_pointwise_scores.csv"
METRICS      = ["Factual Correctness", "Relevance", "Fluency"]
MODEL_NAME   = "Qwen/Qwen2-VL-2B-Instruct"

## ---------------------------------- prompt ----------------------------------
PROMPT_TEMPLATE = """<image>

You are an expert chart-caption evaluator. You will be shown a chart image and a caption generated for it.
Your task is to evaluate the caption on a single criterion using a 1–10 Likert scale.

Criterion: {metric}

Scoring guide:
   1–2  = Very poor   (major errors or completely irrelevant)
   3–4  = Poor        (significant issues, limited accuracy or clarity)
   5–6  = Acceptable  (partially correct or relevant, noticeable gaps)
   7–8  = Good        (mostly accurate, relevant, and well-expressed)
   9–10 = Excellent   (fully accurate, highly relevant, fluent and precise)

Caption: {caption}

Evaluate the caption strictly against the chart image shown.
Respond ONLY with valid JSON, no markdown, no extra text:
{{"score": <integer 1-10>, "explanation": "<one sentence reason>"}}"""

## ---------------------------------- set up ----------------------------------
def mount_drive():
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=True)
        print("Google Drive mounted")
    except ImportError:
        print("Not in Colab")


def load_model():
    print(f"\nLoading {MODEL_NAME} ...")
    from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

    processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"✓ Model loaded on {device}")
    if device == "cuda":
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  GPU memory: {mem:.1f} GB total")
    return model, processor


def load_data():
    with open(JSON_PATH, "r") as f:
        data = json.load(f)
    print(f"Loaded {len(data)} samples from JSON")
    missing_keys = [i for i, d in enumerate(data) if "answer" not in d or "img_id" not in d]
    if missing_keys:
        print(f"  {len(missing_keys)} records missing '{"answer"}' or '{"img_id"}' — will be skipped")
    return data

## ------------------------------- inference ----------------------------------
def parse_response(text: str) -> dict:
    """Extract JSON from model output."""
    text = text.strip()
    # strip markdown fences
    text = re.sub(r"```json\s*", "", text)
    text = re.sub(r"```\s*", "", text)
    # find first {...}
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    # last resort: parse entire string
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"score": None, "explanation": f"PARSE_ERROR: {text[:120]}"}


def score_caption(model, processor, image: Image.Image, caption: str, metric: str) -> dict:

    prompt_text = f"""You are an expert chart evaluator. Rate the following chart caption on a single criterion using a 1-10 Likert scale.

Criterion: {metric}

Scoring guide:
  1–2  = Very poor   (major errors or completely irrelevant)
  3–4  = Poor        (significant issues, limited accuracy or clarity)
  5–6  = Acceptable  (partially correct or relevant, noticeable gaps)
  7–8  = Good        (mostly accurate, relevant, and well-expressed)
  9–10 = Excellent   (fully accurate, highly relevant, fluent and precise)

Caption: {caption}

Evaluate the caption strictly against the chart image shown.
Respond ONLY with valid JSON, no markdown, no extra text:
{{"score": <integer 1-10>, "explanation": "<one sentence reason>"}}"""

    # Qwen2-VL requires messages format with image content block
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]

    # apply chat template to get correctly formatted text
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
        )

    # decode only new tokens
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    response = processor.decode(new_tokens, skip_special_tokens=True)

    return parse_response(response)

## --------------------------- main function ----------------------------------
def run_evaluation():
    mount_drive()
    model, processor = load_model()
    data = load_data()

    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

    rows = []
    skipped = 0

    for sample in tqdm(data, desc="Scoring captions"):
        img_id  = sample.get("img_id")
        caption = sample.get("answer")

        if not img_id or not caption:
            skipped += 1
            continue

        img_path = os.path.join(IMAGES_DIR, f"{img_id}{".png"}")
        if not os.path.exists(img_path):
            print(f"\n  ✗ Image not found, skipping: {img_path}")
            skipped += 1
            continue

        image = Image.open(img_path).convert("RGB")

        row = {"img_id": img_id, "caption": caption}

        for metric in METRICS:
            result = score_caption(model, processor, image, caption, metric)
            col = metric.lower().replace(" ", "_")
            row[f"{col}_score"]       = result.get("score")
            row[f"{col}_explanation"] = result.get("explanation")

        rows.append(row)
        pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)

    # Final save
    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_CSV, index=False)

    print(f"\n{'='*60}")
    print(f"Scored: {len(rows)}   Skipped: {skipped}")
    print(f"Results saved to: {OUTPUT_CSV}")
    print(f"\nScore averages:")
    for metric in METRICS:
        col = metric.lower().replace(" ", "_") + "_score"
        if col in df.columns:
            avg = pd.to_numeric(df[col], errors="coerce").mean()
            print(f"  {metric:25s}: {avg:.2f} / 10.00")
    print('='*60)

    return df



if __name__ == "__main__":
    df = run_evaluation()


Mounted at /content/drive
Google Drive mounted

Loading Qwen/Qwen2-VL-2B-Instruct ...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

✓ Model loaded on cuda
  GPU memory: 15.6 GB total
Loaded 21 samples from JSON


Scoring captions: 100%|██████████| 21/21 [03:54<00:00, 11.15s/it]


Scored: 21   Skipped: 0
Results saved to: /content/drive/MyDrive/EvaltheEvaluators/evaluations/ChartJudge_Suite/Qwen_pointwise_scores.csv

Score averages:
  Factual Correctness      : 7.70 / 10.00
  Relevance                : 7.67 / 10.00
  Fluency                  : 7.81 / 10.00
